In [2]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. DOC DU LIEU
file_path = 'cleaned_data/information/panel_macro_cleaned.csv'
try:
    df = pd.read_csv(file_path)
except FileNotFoundError:
    print(f"Khong tim thay file tai {file_path}.")

# 2. DINH NGHIA CAC CHI SO CAN VE (8 bieu do)
indicators = [
    {'col': 'Economic_Openness_Pct', 'title': '1. Do mo Kinh te (% GDP)', 'ylabel': 'Ty le (%)'},
    {'col': 'GDP_Current_USD', 'title': '2. Quy mo GDP (Ty USD)', 'ylabel': 'Ty USD', 'divide_by': 1e9},
    {'col': 'GDP_Growth_Pct', 'title': '3. Tang truong GDP (%)', 'ylabel': 'Toc do (%)'},
    {'col': 'GDP_GNI_Gap_Pct', 'title': '4. GNI & GDP Gap (% GDP)', 'ylabel': 'Gap (%)'},
    {'col': 'Inflation_CPI_Pct', 'title': '5. Lam phat (CPI %)', 'ylabel': 'Lam phat (%)'},
    {'col': 'FDI_to_GDP_Pct', 'title': '6. Ty trong FDI moi / GDP (%)', 'ylabel': 'FDI / GDP (%)'},
    {'col': 'Lending_Interest_Rate_Pct', 'title': '7. Lai suat cho vay (%)', 'ylabel': 'Lai suat (%)'},
    {'col': 'Remittances_Pct_GDP', 'title': '8. Kieu hoi / GDP (%)', 'ylabel': 'Ty le (%)'}
]

# 3. VE DASHBOARD CHI RIENG VIET NAM
vnm_df = df[df['Country'] == 'VNM'].copy()

fig = make_subplots(
    rows=4, cols=2,
    subplot_titles=[ind['title'] for ind in indicators],
    vertical_spacing=0.08, horizontal_spacing=0.1
)

coords = [(1, 1), (1, 2), (2, 1), (2, 2), (3, 1), (3, 2), (4, 1), (4, 2)]

for i, ind in enumerate(indicators):
    row, col_idx = coords[i]
    col_name = ind['col']

    plot_data = vnm_df.copy()
    if 'divide_by' in ind:
        plot_data[col_name] = plot_data[col_name] / ind['divide_by']

    fig.add_trace(
        go.Scatter(
            x=plot_data['Year'],
            y=plot_data[col_name],
            mode='lines+markers',
            name='VNM',
            line=dict(color='#d62728', width=3.5),
            hovertemplate="<b>VNM</b><br>Nam: %{x}<br>Gia tri: %{y:,.2f}<extra></extra>",
            showlegend=(i == 0)
        ),
        row=row, col=col_idx
    )

    fig.add_hline(y=0, line_dash="dash", line_color="black", opacity=0.5, row=row, col=col_idx)
    fig.update_yaxes(title_text=ind['ylabel'], tickformat=",", row=row, col=col_idx)
    fig.update_xaxes(title_text="Nam", range=[1990, 2024], dtick=5, row=row, col=col_idx)

fig.update_layout(
    height=1800,
    autosize=True,
    margin=dict(l=50, r=20, t=100, b=50),
    title_text="SO LIEU KINH TE VI MO: VIET NAM",
    title_font=dict(size=24, color='black'),
    title_x=0.05,
    hovermode="x unified",
    template="plotly_white",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    )
)

fig.show(config={'responsive': True})
fig.write_html("So_lieu_VNM.html")

In [3]:
import plotly.express as px

# 1. Chuẩn bị dữ liệu và phân nhóm (Giữ nguyên như của bạn)
latest_year = int(df['Year'].max())
focus_countries = ['SGP', 'IRL', 'USA', 'IND', 'DEU', 'CHN', 'JPN', 'VNM', 'THA', 'PHL', 'ZAF']

group_map = {
    'SGP': 'Siêu mở',
    'IRL': 'Siêu mở',
    'USA': 'Thị trường nội địa',
    'IND': 'Thị trường nội địa',
    'DEU': 'Công nghiệp xuất khẩu',
    'CHN': 'Công nghiệp xuất khẩu',
    'JPN': 'Công nghiệp xuất khẩu',
    'VNM': 'Việt Nam', 
    'THA': 'Mô hình trung gian',
    'PHL': 'Mô hình trung gian',
    'ZAF': 'Mô hình trung gian',
}

color_map = {
    'Siêu mở': '#00BFFF',               
    'Thị trường nội địa': '#8B4513',    
    'Công nghiệp xuất khẩu': '#808080', 
    'Mô hình trung gian': '#FF8C00',    
    'Việt Nam': '#E60000' 
}

bubble_df = df[(df['Year'] == latest_year) & (df['Country'].isin(focus_countries))].copy()
bubble_df['GDP_Billion_USD'] = bubble_df['GDP_Current_USD'] / 1e9
bubble_df['Group'] = bubble_df['Country'].map(group_map)

# Điền 0 cho các giá trị NaN của GDP per capita để tránh lỗi khi vẽ size
bubble_df['GDP_Per_Capita_USD'] = bubble_df['GDP_Per_Capita_USD'].fillna(0)

# 2. Vẽ biểu đồ bằng Plotly Express
fig_bubble = px.scatter(
    bubble_df,
    x='GDP_Billion_USD',
    y='Economic_Openness_Pct',
    size='GDP_Per_Capita_USD',    # Tự động tạo Size Legend
    color='Group',                # Tự động tạo Color Legend (có thể click lọc)
    color_discrete_map=color_map, # Áp dụng màu sắc của bạn
    text='Country',
    hover_name='Country',
    size_max=60,                  # Điều chỉnh kích thước bong bóng lớn nhất
    title=f"BIỂU ĐỒ BONG BÓNG: ĐỘ MỞ VS QUY MÔ (NĂM {latest_year})",
    labels={
        'GDP_Billion_USD': 'GDP (Tỷ USD)',
        'Economic_Openness_Pct': 'Độ mở kinh tế (% GDP)',
        'Group': 'Nhóm quốc gia',
        'GDP_Per_Capita_USD': 'GDP/Người (USD)'
    }
)

# 3. Tùy chỉnh hiển thị (Label, Tooltip, Layout)
fig_bubble.update_traces(
    textposition='top center',
    marker=dict(line=dict(width=1, color='white')),
    hovertemplate=(
        "<b>%{hovertext}</b><br>"
        "GDP: %{x:,.1f} Tỷ USD<br>"
        "Độ mở: %{y:,.1f} %<br>"
        "GDP/Người: %{marker.size:,.0f} USD<extra></extra>" 
        # (Lưu ý: trong px, marker.size lưu giá trị thực tế truyền vào)
    )
)

fig_bubble.update_layout(
    template="plotly_white",
    height=700,
    margin=dict(l=60, r=40, t=80, b=60),
    legend=dict(
        title_font_family="Arial",
        font=dict(size=12)
    )
)

# Thêm tuỳ chỉnh Log cho trục X nếu bong bóng của Mỹ và Trung Quốc làm biểu đồ bị dồn (Bỏ comment nếu muốn dùng)
fig_bubble.update_xaxes(type="log", title="GDP (Tỷ USD) - Thang đo Logarit")

fig_bubble.show(config={'responsive': True})
fig_bubble.write_html("bubble_chart_openness_gdp.html")

In [9]:
df = pd.read_csv('cleaned_data/VNM_macro_cleaned.csv')

fig = make_subplots(specs=[[{"secondary_y": True}]])

# Thêm FDI Inflows dưới dạng cột
fig.add_trace(
    go.Bar(x=df['Year'], y=df['FDI_Inflows_USD'], name="Vốn FDI (USD)", marker_color="#2FA1FF"),
    secondary_y=False,
)

# Thêm Xuất khẩu dưới dạng đường
fig.add_trace(
    go.Scatter(x=df['Year'], y=df['Exports_USD'], name="Kim ngạch Xuất khẩu (USD)", line=dict(color="#FF4A4A", width=3)),
    secondary_y=True,
)

fig.update_layout(
    title_text="<b>Tương quan giữa dòng vốn FDI và Tăng trưởng Xuất khẩu Việt Nam (1990-2024)</b>",
    hovermode="x unified"
)

fig.update_xaxes(title_text="Năm")
fig.update_yaxes(title_text="FDI Inflows (Current USD)", secondary_y=False)
fig.update_yaxes(title_text="Exports (Current USD)", secondary_y=True)

fig.show()